# 03. Model and Cost Comparison

**Purpose:** This notebook teaches practical model trade-offs: balancing **cost**, **speed**, and **accuracy**.

**Workflow:**
1. Reuse your best prompt from Notebook 02.
2. Run the same prompt on **two models**:
   - `gemini-2.5-flash` (fast/cheap)
   - `gemini-2.5-pro` (more accurate/more costly)
3. Compare results and discuss trade-offs.


## Step 1: Install Dependencies

In [ ]:
!git clone https://github.com/alxefremov/esmt-workshop.git
%pip install -q -r esmt-workshop/requirements.txt


## Step 2: Load Shared Utilities

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Resolve repository root both for local runs and Google Colab.
PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/esmt-workshop')]:
    if (candidate / 'src' / 'esmt_workshop').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Project root not found. Run this notebook from the ESMT_Workshop repository.'
    )

# Make local package importable inside notebook execution context.
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from esmt_workshop.evaluation import evaluate_predictions, save_evaluation_report
from esmt_workshop.experiment_logging import load_experiment_history, log_experiment_run
from esmt_workshop.student_api import process_batch_addresses, process_single_address
print('PROJECT_ROOT =', PROJECT_ROOT)


## Step 3: Student Identity Parameter

In [ ]:
from google.colab import auth

auth.authenticate_user()

gcloud_email = !gcloud config get-value account
print(f"Authenticated as: {gcloud_email[0] if gcloud_email else 'Unknown'}")
os.environ['WORKSHOP_EMAIL'] = gcloud_email[0] if gcloud_email else os.environ.get('WORKSHOP_EMAIL', '')

# Students only provide email; proxy endpoint details are managed by organizers.
STUDENT_EMAIL = os.getenv('WORKSHOP_EMAIL', 'student@example.com')
print('STUDENT_EMAIL =', STUDENT_EMAIL)


## Step 4: Configuration, Prompt, and Data

In [ ]:
import time

# Notebook metadata (used for saved artifacts/logs)
NOTEBOOK_NAME = '03_model_and_cost_comparison'

# Keep True for offline demo; set False when organizer-managed proxy is available.
MOCK_MODE = True

# Model choices (organizers may override via env vars)
FLASH_MODEL = os.getenv('WORKSHOP_BASELINE_MODEL', 'gemini-2.5-flash')
PRO_MODEL = os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro')

# Generation controls (keep consistent for fair comparison)
TEMPERATURE = 0.0
TOP_P = 0.95
TOP_K = 40
MAX_TOKENS = None  # keep toolkit default unless you need to cap
MAX_WORKERS = 8

# Prompt is editable directly in the notebook (reuse your best prompt from Notebook 02).
USE_CUSTOM_PROMPT = True
PROMPT_LABEL = 'student_prompt'
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Town Name should be city/locality only.
- Postal Code should include only postal token(s).
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''
custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None

# Load labeled dev set for scoring.
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

print('FLASH_MODEL =', FLASH_MODEL)
print('PRO_MODEL   =', PRO_MODEL)
print('dev_df shape =', dev_df.shape)


## Step 5: Run Flash Model (Baseline Stage)

In [ ]:
# Step 5: Run Baseline Stage with Flash model (uses baseline notebook logic)
STAGE_NAME = 'baseline'

print('Running baseline stage with Flash model:', FLASH_MODEL)
t0 = time.perf_counter()

pred_flash = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage=STAGE_NAME,
    model=FLASH_MODEL,
    country_model=None,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    top_k=TOP_K,
    max_tokens=MAX_TOKENS,
    use_guardrails=False,
    custom_prompt_template=custom_prompt,
    kb_csv_path=str(PROJECT_ROOT / 'data/input/address_formats.csv'),
    max_workers=MAX_WORKERS,
    mock_mode=MOCK_MODE,
)

runtime_flash = time.perf_counter() - t0
report_flash = evaluate_predictions(pred_flash, dev_df, email=STUDENT_EMAIL)

print(report_flash['summary'])
report_flash['field_metrics']


## Step 6: Run Pro Model (Advanced Stage)

In [ ]:
# Prompt is editable directly in the notebook (no external prompt files).
USE_CUSTOM_PROMPT = True
PROMPT_TEMPLATE = '''You are an AML address parser.

Input address:
{address}

Return strict JSON only with schema:
{schema}

Rules:
- Town Name should be city/locality only.
- Postal Code should include only postal token(s).
- Country Code must be ISO alpha-2 uppercase.
- Use empty strings when unknown.
'''

custom_prompt = PROMPT_TEMPLATE if USE_CUSTOM_PROMPT else None
dev_df = pd.read_csv(PROJECT_ROOT / 'data/workshop/dev_labeled.csv', dtype=str).fillna('')

pred_df = process_batch_addresses(
    dev_df,
    email=STUDENT_EMAIL,
    stage='advanced',
    model=os.getenv('WORKSHOP_ADVANCED_MODEL', 'gemini-2.5-pro'),
    temperature=0.0,
    top_p=0.95,
    top_k=40,
    custom_prompt_template=custom_prompt,
    max_workers=8,
)
report = evaluate_predictions(pred_df, dev_df, email=STUDENT_EMAIL)

print(report['summary'])
report['field_metrics']


## Step 7: Compare Results

In [ ]:
comparison_df = pd.DataFrame([
    {'stage': 'baseline', 'model': FLASH_MODEL, 'micro_accuracy': report_flash.get('summary', {}).get('micro_accuracy', 0.0), 'runtime_sec': runtime_flash},
    {'stage': 'advanced', 'model': PRO_MODEL,   'micro_accuracy': report_pro.get('summary', {}).get('micro_accuracy', 0.0),   'runtime_sec': runtime_pro},
])
comparison_df


## Step 8: Save Artifacts

In [ ]:
# Save predictions and evaluation files for both runs.
out_dir = PROJECT_ROOT / 'outputs'
out_dir.mkdir(parents=True, exist_ok=True)

pred_flash_path = out_dir / '03_model_and_cost_flash_predictions.csv'
pred_flash.to_csv(pred_flash_path, index=False)

pred_pro_path = out_dir / '03_model_and_cost_pro_predictions.csv'
pred_pro.to_csv(pred_pro_path, index=False)

report_flash_dir = out_dir / 'report_03_model_and_cost_flash'
if report_flash.get('summary'):
    save_evaluation_report(report_flash, report_flash_dir)

report_pro_dir = out_dir / 'report_03_model_and_cost_pro'
if report_pro.get('summary'):
    save_evaluation_report(report_pro, report_pro_dir)

print('Saved predictions (flash):', pred_flash_path)
print('Saved predictions (pro):', pred_pro_path)
if report_flash.get('summary'):
    print('Saved report (flash):', report_flash_dir)
if report_pro.get('summary'):
    print('Saved report (pro):', report_pro_dir)


## Discussion
- Where does the Pro model improve most? Inspect `field_metrics` and mismatches.
- If Pro is slower/more costly, when is Flash the better business choice?
